# Computer Organisation & Architecture — CO1

**2310221L.CO.1** — *Explicate the architecture and instruction set of the 80386
microprocessor.* **[L2]**

---

## Architecture answers two questions

1. **What parts exist**, and what is each for?
2. **What can a program ask for** — the instruction set?

![the datapath](diagrams/coa-datapath.png)

---

## The parts

| Register | For |
|---|---|
| R0 – R7 | general purpose |
| PC | which instruction comes next |
| IR | the one being carried out now |
| SP | how deep the call stack is |
| MAR / MDR | the address being accessed, and the value going to or from memory |

MAR and MDR exist because a processor cannot reach memory in one motion — it puts an
address where memory can see it, then collects what comes back.

```cpp
class Word {
private:
    u16 value_;
public:
    static const int BITS = 16;

    Word operator+(const Word& o) const { return Word(static_cast<u16>(value_ + o.value_)); }
    Word operator-(const Word& o) const { return Word(static_cast<u16>(value_ - o.value_)); }
    Word operator&(const Word& o) const { return Word(static_cast<u16>(value_ & o.value_)); }

    bool bit(int i) const { return ((value_ >> i) & 1u) != 0; }
    bool msb()      const { return bit(BITS - 1); }        // the sign bit
};
```
<sub>src/core/Word.h:21</sub>

16 bits, so a register holds 0 to 65,535 — and the cast is why `0 - 1` gives 65,535, not −1.

```cpp
Word read()        { readThisCycle_ = true;  return value_; }
void write(Word v) { value_ = v; wroteThisCycle_ = true; }
```
<sub>src/core/RegisterFile.h:36</sub>

Each register records that it was touched, which is how the display highlights what took
part.

---

## The ALU and its flags

![the ALU and its flags](diagrams/coa-alu-flags.png)

```cpp
enum AluOp {
    ALU_ADD, ALU_SUB, ALU_AND, ALU_OR, ALU_XOR, ALU_NOT,
    ALU_SHL, ALU_SHR, ALU_CMP, ALU_INC, ALU_DEC,
    ALU_PASS_A, ALU_PASS_B, ALU_NONE
};

struct Flags {
    bool zero;      // result was 0
    bool carry;     // carry out of the MSB
    bool overflow;  // signed overflow
    bool negative;  // MSB of the result is set
};

AluResult execute(AluOp op, Word a, Word b);
```
<sub>src/core/ALU.h:21</sub>

**The flags are where a calculator becomes a computer.** `CMP` subtracts, throws the
answer away, and keeps only the flags; `JNZ` reads them and decides whether to jump. That
pair is what a loop is made of.

```cpp
class Memory {
    ds::HashMap<unsigned int, u16> cells_;   // address -> value, sparse
public:
    static const unsigned int SIZE = 0x10000;   // 64K addressable words
    Word read(unsigned int address);
    void write(unsigned int address, Word value);
};
```
<sub>src/core/Memory.h:22</sub>

All of these share **one data bus**, so only one may drive it at a time — which is why a
control unit is needed to say whose turn it is.

---

## The instruction set

Fourteen instructions, three addressing modes.

| Instruction | Does |
|---|---|
| `LOAD Rd, #n` | a fixed number into a register |
| `LOADM Rd, [addr]` / `STORE Rs, [addr]` | memory in, memory out |
| `MOV Rd, Rs` | register to register |
| `ADD` `SUB` `AND` `OR` `XOR` `Rd, Rs` | arithmetic and logic |
| `CMP Rd, Rs` | compare — sets flags, writes nothing |
| `INC` `DEC` `Rd` | ±1 |
| `JMP` `JZ` `JNZ` `addr` | jump, always or on the zero flag |
| `OUT Rs` · `HLT` | print · stop |

| Mode | Example | Operand is |
|---|---|---|
| immediate | `LOAD R0, #5` | in the instruction |
| register | `ADD R0, R1` | in a register |
| direct | `STORE R0, [0x40]` | in memory |

```cpp
class Instruction {
protected:
    std::string mnemonic_;
    int         rd_;        // destination register, -1 if unused
    int         rs_;        // source register, -1 if unused
    Word        operand_;   // immediate value or address
public:
    virtual ~Instruction() {}
    virtual void           execute(CPU& cpu) = 0;
    virtual ControlSignals signals() const   = 0;
};
```
<sub>src/core/Instruction.h:65</sub>

---

## One instruction, four stages

![the instruction cycle](diagrams/coa-instruction-cycle.png)

```cpp
case STAGE_FETCH: {
    unsigned int addr = regs_.pc().peek().raw();
    regs_.mar().write(Word(static_cast<u16>(addr)));
    current_ = instructionAt(addr);
    signals_.pcOut = true;  signals_.marLoad = true;
    signals_.memRead = true;  signals_.irLoad = true;
    stage_ = STAGE_DECODE;
    break;
}

case STAGE_DECODE: {
    signals_ = current_->signals();        // ask the instruction what it needs
    stage_ = STAGE_EXECUTE;
    break;
}

case STAGE_EXECUTE: {
    current_->execute(*this);              // the instruction does its work
    stage_ = STAGE_WRITEBACK;
    break;
}

case STAGE_WRITEBACK: {
    if (current_ != 0 && !halted_ && !branchTaken_) {
        regs_.pc().write(regs_.pc().peek() + Word(1));
        signals_.pcInc = true;
    }
    stage_ = STAGE_FETCH;
    break;
}
```
<sub>src/core/CPU.cpp:128 — condensed</sub>

**One call advances one stage, not one instruction.** And note writeback: the PC only
advances if no branch happened, because a jump works by writing a different number into
it.

---

## In one minute

1. **Architecture = what parts exist, and what a program may ask for.**
2. Eight registers plus PC, IR, SP, MAR, MDR; a 16-bit word; an ALU; memory; one shared bus.
3. **The flags decide.** `CMP` sets them, `JNZ` reads them — that pair is a loop.
4. Fourteen instructions, three addressing modes: immediate, register, direct.
5. **Four stages, steppable one at a time** — so the cycle is watched, not memorised.